In [1]:
import pyarrow as pa
import pyarrow.csv as pv
import pyarrow.compute as pc
import pyarrow.parquet as pq
import re

# -------- Configuração do schema desejado --------
schema = pa.schema([
    pa.field("nu_cpf_paciente", pa.string()),
    pa.field("nu_cns_paciente", pa.string()),
    pa.field("co_sigtap", pa.string()),
    pa.field("co_cbo", pa.string()),
    pa.field("sg_uf_estab_executante", pa.string()),
    pa.field("co_municipio_estab_executante", pa.string()),
    pa.field("co_cnes_estab_executante", pa.string()),
    pa.field("data_solicitacao", pa.date32()),
    pa.field("data_autorizacao", pa.date32()),
    pa.field("data_execucao", pa.date32()),
    pa.field("st_vida_paciente", pa.bool_()),
    pa.field("st_solicitacao", pa.string()),
    pa.field("id_registro_sistema_origem", pa.string()),
    pa.field("ds_sistema_origem", pa.string()),
])

# -------- Funções de limpeza/normalização --------
def only_digits_array(arr: pa.Array) -> pa.Array:
    # Remove tudo que não é dígito
    return pc.replace_substring_regex(arr, pattern=r"\D+", replacement="")

def pad_left_array(arr: pa.Array, width: int) -> pa.Array:
    # Preenche com zeros à esquerda até width; preserva nulos
    arr = pc.utf8_lpad(arr, width=width, padding="0")
    # Opcional: quando string vazia virar nulo
    arr = pc.if_else(pc.equal(arr, pa.scalar("0"*width)),
                     pc.cast(pc.if_else(pc.equal(arr, pa.scalar("0"*width)), pa.scalar(None), arr), pa.string()),
                     arr)
    return arr

def normalize_id(arr: pa.Array, width: int, allow_null=True) -> pa.Array:
    # Mantém somente dígitos, corta/pad para o comprimento alvo
    arr = pc.cast(arr, pa.string())
    arr = only_digits_array(arr)
    # Se vazio e allow_null, vira nulo
    if allow_null:
        arr = pc.if_else(pc.equal(arr, pa.scalar("")),
                         pa.scalar(None, type=pa.string()),
                         arr)
    # Trunca se passar do tamanho
    arr = pc.utf8_slice_codeunits(arr, start=0, stop=width)
    # Pad à esquerda
    arr = pc.utf8_lpad(arr, width=width, padding="0")
    return arr

def parse_date_ddmmyyyy(arr: pa.Array) -> pa.Array:
    arr = pc.cast(arr, pa.string())
    # gera timestamp
    parsed = pc.strptime(arr, format="%d/%m/%Y", unit="s", error_is_null=True)
    # converte para date32 (só a parte da data)
    return pc.cast(parsed, pa.date32())


def to_bool(arr: pa.Array) -> pa.Array:
    # Aceita "TRUE"/"FALSE", "true"/"false", "1"/"0" e nulos
    arr = pc.cast(arr, pa.string())
    lower = pc.ascii_lower(arr)
    true_mask = pc.is_in(lower, value_set=pa.array(["true", "1"]))
    false_mask = pc.is_in(lower, value_set=pa.array(["false", "0"]))
    # default: nulo quando não reconhecido
    return pc.if_else(true_mask, pa.scalar(True), pc.if_else(false_mask, pa.scalar(False), pa.scalar(None, type=pa.bool_())))

def normalize_text(arr: pa.Array) -> pa.Array:
    arr = pc.cast(arr, pa.string())
    # Remove espaços em branco (default é " \t\r\n")
    return pc.utf8_trim(arr, options=pc.TrimOptions(characters=" \t\r\n"))


# -------- Conversão streaming CSV -> Parquet --------
def csv_to_parquet_streaming(
    csv_path: str,
    parquet_path: str,
    chunk_size_bytes: int = 64 * 1024 * 1024,
    compression: str = "zstd",
):
    # Opções de leitura
    read_options = pv.ReadOptions(block_size=chunk_size_bytes)
    parse_options = pv.ParseOptions(delimiter=";", quote_char='"')
    convert_options = pv.ConvertOptions(
        column_types={col: pa.string() for col in schema.names}
    )

    # Leitura streaming
    with pv.open_csv(csv_path,
                     read_options=read_options,
                     parse_options=parse_options,
                     convert_options=convert_options) as reader:
        writer = None
        try:
            for tb in reader:
                # Transformações
                dt_solic = parse_date_ddmmyyyy(tb["data_solicitacao"])
                dt_aut = parse_date_ddmmyyyy(tb["data_autorizacao"])
                dt_exec = parse_date_ddmmyyyy(tb["data_execucao"])
                vida = to_bool(tb["st_vida_paciente"])
                st_sol = normalize_text(tb["st_solicitacao"])
                id_origem = normalize_text(tb["id_registro_sistema_origem"])
                ds_origem = normalize_text(tb["ds_sistema_origem"])

                out = pa.table({
                    "nu_cpf_paciente": tb["nu_cpf_paciente"],
                    "nu_cns_paciente": tb["nu_cns_paciente"],
                    "co_sigtap": tb["co_sigtap"],
                    "co_cbo": tb["co_cbo"],
                    "sg_uf_estab_executante": tb["sg_uf_estab_executante"],
                    "co_municipio_estab_executante": tb["co_municipio_estab_executante"],
                    "co_cnes_estab_executante": tb["co_cnes_estab_executante"],
                    "data_solicitacao": dt_solic,
                    "data_autorizacao": dt_aut,
                    "data_execucao": dt_exec,
                    "st_vida_paciente": vida,
                    "st_solicitacao": st_sol,
                    "id_registro_sistema_origem": id_origem,
                    "ds_sistema_origem": ds_origem,
                }, schema=schema)

                if writer is None:
                    writer = pq.ParquetWriter(parquet_path, schema=schema,
                                              compression=compression, use_dictionary=True)
                writer.write_table(out)
        finally:
            if writer is not None:
                writer.close()
# --------- Executar ---------
if __name__ == "__main__":
    csv_path = "base/tb_ra_202511241538.csv"            # substitua pelo seu arquivo
    parquet_path = "base/tb_ra_202511241538.parquet"    # nome do parquet final
    csv_to_parquet_streaming(csv_path, parquet_path)
    print("Conversão concluída:", parquet_path)


Conversão concluída: base/tb_ra_202511241538.parquet


In [5]:
import pyarrow as pa
import pyarrow.csv as pv
import pyarrow.compute as pc
import pyarrow.parquet as pq
import re

# ==============================================================================
# 1. Configuração do schema desejado para o arquivo SIA.csv
# ==============================================================================
# Baseado nas colunas do SQL:
schema_sia = pa.schema([
    pa.field("CPF_PAC", pa.string()),
    pa.field("CNS_PAC", pa.string()),
    pa.field("COD_SIGTAP_PROCEDIMENTO", pa.string()),
    pa.field("CBO", pa.string()),
    pa.field("UF_DESC_ATEND", pa.string()),
    pa.field("IBGE_ATEND", pa.string()),
    pa.field("CNES_ATEND", pa.string()),
    pa.field("DT_CMP_FORMATADA", pa.string()), # Mês/Ano (MM/YYYY) - manter como string para simplificar
    pa.field("DATA_SOLICITACAO", pa.date32()), # DD/MM/YYYY -> Date
    pa.field("DATA_AUTORIZACAO", pa.date32()),  # DD/MM/YYYY -> Date
    pa.field("DATA_INICIO", pa.date32()),       # DD/MM/YYYY -> Date
    pa.field("DATA_FINAL", pa.date32()),        # DD/MM/YYYY -> Date
    pa.field("CO_APA_NUM", pa.string()), # Número da APA
])

# ==============================================================================
# 2. Funções de limpeza/normalização (Reutilizadas e Adaptadas)
# ==============================================================================

def only_digits_array(arr: pa.Array) -> pa.Array:
    """Remove tudo que não é dígito."""
    return pc.replace_substring_regex(arr, pattern=r"\D+", replacement="")

def parse_date_ddmmyyyy(arr: pa.Array) -> pa.Array:
    """Converte strings no formato DD/MM/YYYY para pa.date32."""
    arr = pc.cast(arr, pa.string())
    # gera timestamp
    parsed = pc.strptime(arr, format="%d/%m/%Y", unit="s", error_is_null=True)
    # converte para date32 (só a parte da data)
    return pc.cast(parsed, pa.date32())

def normalize_text(arr: pa.Array) -> pa.Array:
    """Remove espaços em branco no início/fim e converte para string."""
    arr = pc.cast(arr, pa.string())
    return pc.utf8_trim(arr, options=pc.TrimOptions(characters=" \t\r\n"))


# ==============================================================================
# 3. Conversão streaming CSV -> Parquet (Função Adaptada)
# ==============================================================================
def csv_to_parquet_streaming_sia(
    csv_path: str,
    parquet_path: str,
    target_schema: pa.Schema, # Agora aceita o schema como argumento
    chunk_size_bytes: int = 64 * 1024 * 1024,
    compression: str = "zstd",
):
    """
    Processa um CSV por chunks e salva como Parquet.
    Adapta as transformações para o schema SIA.
    """
    # Opções de leitura
    read_options = pv.ReadOptions(block_size=chunk_size_bytes)
    parse_options = pv.ParseOptions(delimiter=";", quote_char='"')
    # Importante: ler todas as colunas como string primeiro para depois aplicar as transformações
    convert_options = pv.ConvertOptions(
        column_types={col: pa.string() for col in target_schema.names}
    )

    # Leitura streaming
    with pv.open_csv(csv_path,
                     read_options=read_options,
                     parse_options=parse_options,
                     convert_options=convert_options) as reader:
        writer = None
        try:
            for tb in reader:
                # Renomear as colunas lidas para o esquema de destino, se necessário,
                # e aplicar as transformações de data/texto.
                
                # Campos de Data
                dt_solic = parse_date_ddmmyyyy(tb["DATA_SOLICITACAO"])
                dt_aut = parse_date_ddmmyyyy(tb["DATA_AUTORIZACAO"])
                dt_inicio = parse_date_ddmmyyyy(tb["DATA_INICIO"])
                dt_final = parse_date_ddmmyyyy(tb["DATA_FINAL"])

                # Campos de Texto/ID (aplica trim)
                cpf_pac = normalize_text(tb["CPF_PAC"])
                cns_pac = normalize_text(tb["CNS_PAC"])
                co_sigtap = normalize_text(tb["COD_SIGTAP_PROCEDIMENTO"])
                co_cbo = normalize_text(tb["CBO"])
                sg_uf = normalize_text(tb["UF_DESC_ATEND"])
                co_municipio = normalize_text(tb["IBGE_ATEND"])
                co_cnes = normalize_text(tb["CNES_ATEND"])
                dt_cmp = normalize_text(tb["DT_CMP_FORMATADA"])
                co_apa_num = normalize_text(tb["CO_APA_NUM"])

                # Monta a nova tabela com as colunas transformadas
                out = pa.table({
                    "CPF_PAC": cpf_pac,
                    "CNS_PAC": cns_pac,
                    "COD_SIGTAP_PROCEDIMENTO": co_sigtap,
                    "CBO": co_cbo,
                    "UF_DESC_ATEND": sg_uf,
                    "IBGE_ATEND": co_municipio,
                    "CNES_ATEND": co_cnes,
                    "DT_CMP_FORMATADA": dt_cmp,
                    "DATA_SOLICITACAO": dt_solic,
                    "DATA_AUTORIZACAO": dt_aut,
                    "DATA_INICIO": dt_inicio,
                    "DATA_FINAL": dt_final,
                    "CO_APA_NUM": co_apa_num,
                }, schema=target_schema)

                if writer is None:
                    # Inicializa o escritor Parquet com o schema do SIA
                    writer = pq.ParquetWriter(parquet_path, schema=target_schema,
                                              compression=compression, use_dictionary=True)
                writer.write_table(out)
        finally:
            if writer is not None:
                writer.close()
                
# ==============================================================================
# 4. Executar para o arquivo SIA.csv
# ==============================================================================
if __name__ == "__main__":
    # --- Execução do SIA.csv ---
    csv_path_sia = "base\SIA.csv"  # Arquivo de origem
    parquet_path_sia = "base\SIA.parquet"  # Nome do parquet final
    
    # Você pode manter a execução do arquivo original se desejar
    # csv_path_ra = "base/tb_ra_202511241538.csv"
    # parquet_path_ra = "base/tb_ra_202511241538.parquet"
    # csv_to_parquet_streaming_original(csv_path_ra, parquet_path_ra, schema)
    
    print(f"Iniciando conversão de {csv_path_sia} para {parquet_path_sia}...")
    try:
        csv_to_parquet_streaming_sia(csv_path_sia, parquet_path_sia, target_schema=schema_sia)
        print("✅ Conversão SIA concluída:", parquet_path_sia)
    except Exception as e:
        print(f"❌ Erro durante a conversão do SIA.csv: {e}")

<>:130: SyntaxWarning: invalid escape sequence '\S'
<>:131: SyntaxWarning: invalid escape sequence '\S'
<>:130: SyntaxWarning: invalid escape sequence '\S'
<>:131: SyntaxWarning: invalid escape sequence '\S'
C:\Users\Datasus\AppData\Local\Temp\ipykernel_10808\1021472288.py:130: SyntaxWarning: invalid escape sequence '\S'
  csv_path_sia = "base\SIA.csv"  # Arquivo de origem
C:\Users\Datasus\AppData\Local\Temp\ipykernel_10808\1021472288.py:131: SyntaxWarning: invalid escape sequence '\S'
  parquet_path_sia = "base\SIA.parquet"  # Nome do parquet final


Iniciando conversão de base\SIA.csv para base\SIA.parquet...
✅ Conversão SIA concluída: base\SIA.parquet
